> Experiments with [data](../trapallada/data.py)  module

In [1]:
import pathlib

import yaml

import jax.random as jr

import trapallada.data
import trapallada.model.nn
import trapallada.util
import trapallada.core

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Directory containing the data

In [3]:
# # input_directory = pathlib.Path.cwd().parent / 'samples' / 'data' / 'mini_rate'
# input_directory = pathlib.Path.cwd().parent / 'data' / 'rates_large_infs'
# assert input_directory.exists()
# input_directory

A parameters file that specifies an experiment

In [4]:
parameters_file = pathlib.Path.cwd().parent / 'parameters.yaml'
assert parameters_file.exists()

It is read into a `dic`

In [5]:
with parameters_file.open() as f:
    parameters = yaml.safe_load(f)

In [6]:
%pycat {parameters_file}

PRNG seeds:
  model initialization:
    decoder: 43
    encoder: 42
  sampling:
    training set: 14
    validation set: 15
  training dataloader: 43
  training-validation splitting: 42
loss:
  x_sd: 0.15 # <---- FFT
model: &model
  # channels: [12, 12, 12]
  channels:
  - 8
  - 16
  - 32
  dim_z: 40
  kernel_sizes: # <---- FFT (due to redundancy)
  - 64
  - 32
  - 16
  # stride: 2
  stride:
  - 2
  - 2
  - 2
data:
    fft: false # <---- FFT
training:
  batch_size: 32
  dl_rng_key: 43
  learning_rate: 3.0e-05
  n_epochs: 2
  
  # ratio of the dataset to use for validation
  train_test_split:
    test_size: 0.2
  
  # whether or not to ignore non "good time intervals" (GTIs) in the loss function; it only
  # makes sense when working with time-domain signals, and will be ignored for frequency-domain ones
  mask_non_gti_intervals: true
VAE:
  class: VAE
  encoder:
    class: Encoder
    parameters:
      << : [*model]
  decoder:
    class: Decoder # <---- FFT (NonNegativeDecoder/NonNegati

# sample_dataset_parameters

It returns the files making up a *sample* dataset (that are expected by `trapallada.data.SignalsDataset`):


- data itself, in the form of an `Array`

- mask, another `Array`

- metadata in a *Pandas* `DataFrame`

In [7]:
trapallada.data.sample_dataset_parameters()

{'signals_filename': PosixPath('/home/manu/Sync/git/trapallada/data/rates_large_infs/signals.pt'),
 'metadata_filename': PosixPath('/home/manu/Sync/git/trapallada/data/rates_large_infs/metadata.parquet'),
 'masks_filename': PosixPath('/home/manu/Sync/git/trapallada/data/rates_large_infs/masks.pt')}

It is turned into a *PyTorch* `Dataset`. By default, *windows* are **not** normalized.

In [8]:
# ds = trapallada.data.SignalsDataset(signals_file, metadata_file, masks_file, fft=True)
ds = trapallada.data.SignalsDataset(**trapallada.data.sample_dataset_parameters(), fft=parameters['data']['fft'])
len(ds), ds[0][1].shape

(30856, torch.Size([5, 225]))

Inside the `Dataset`, there are `array`s

In [9]:
ds.signals.shape, ds.masks.shape

(torch.Size([30856, 5, 225]), (30856, 225))

In [10]:
to_validation = [e + '.FTZ' for e in trapallada.data.Labels().signals_without_prefix]
to_validation

['P0605950101PNX000SRCTSR8001.FTZ',
 'P0840380201PNX000SRCTSR8004.FTZ',
 'P0690744401PNX000SRCTSR8002.FTZ',
 'P0148850101PNX000SRCTSR8004.FTZ',
 'P0740290301PNX000SRCTSR8002.FTZ',
 'P0720173901PNX000SRCTSR8001.FTZ']

In [11]:
parameters['training']['train_test_split']

{'test_size': 0.2}

In [12]:
has_label = ds.metadata_df['source'].isin([e + '.FTZ' for e in trapallada.data.Labels().signals_without_prefix])
has_label

0        False
1        False
2        False
3        False
4        False
         ...  
30851    False
30852    False
30853    False
30854    False
30855    False
Name: source, Length: 30856, dtype: bool

In [13]:
# np.flatnonzero(has_label)
has_label[has_label].index

Index([ 8222,  8224,  8226,  8228,  8230,  8232,  8234,  8236,  8238,  8240,
        8242,  8244,  8246,  8248, 12284, 12288, 12292, 12296, 12300, 12304,
       12308, 12312, 12316, 12320, 12324, 12328, 12332, 12336, 12340, 12344,
       12348, 12352, 12356, 12360, 12364, 12368, 12372, 12376, 12380, 12384,
       12388, 12392, 12396, 16827, 16828, 16829, 16830, 16831, 16832, 16833,
       16834, 16835, 16836, 16837, 16838, 16839, 16840, 26781, 26782, 26783,
       26784, 26785, 26786, 26787, 26788, 26789, 26790, 26791, 26792, 26793,
       26794, 26795, 26796, 26797, 26798, 26799, 26800, 26801, 26802, 26803,
       26804, 26805],
      dtype='int64')

In [14]:
ds.metadata_df.columns

Index(['start', 'end', 'tar file', 'source'], dtype='object')

# dataloaders_from_dataset

`DataLoaders` are derived from the `Dataset`

In [15]:
train_dl, validation_dl = trapallada.data.dataloaders_from_dataset(
    ds,
    parameters['training']['train_test_split'] | dict(random_state=parameters['PRNG seeds']['training-validation splitting']),
    parameters['training']['batch_size'], parameters['PRNG seeds']['training dataloader'])


How many samples per batch?

In [16]:
batch = next(iter(train_dl))
len(batch)

3

In the first batch
- the *index component* 

In [17]:
batch[0]

tensor([22405, 30796, 21653,  7324, 10512, 12216, 22197, 25170, 28879,  2016,
        14391, 11663,  9567,   707,  6102,  7610, 23979, 29309, 29556, 20203,
        20197,  1890, 27996, 28402, 19818, 30835,  3139, 16120,   926,  1953,
         2452, 22121])

- the signals

In [18]:
batch[1].shape

torch.Size([32, 5, 225])

- the masks

In [19]:
batch[2].shape

torch.Size([32, 225])

- the labels (or `None`, if unsupervised)

The input size is expected along with the parameters

In [20]:
parameters['VAE']['encoder']['parameters']['input_size'] = trapallada.util.input_size_from_dataset(ds)

In [21]:
x = batch[1][0].numpy()
encoder = trapallada.model.nn.Encoder(
    **parameters['VAE']['encoder']['parameters'],
    key=jr.PRNGKey(parameters['PRNG seeds']['model initialization']['encoder']))
encoder.debug_call(x)
print(encoder)

after equinox.nn._conv: (8, 81)
after jax._src.nn.functions: (8, 81)
after equinox.nn._normalisation: (8, 81)
after equinox.nn._conv: (16, 25)
after jax._src.nn.functions: (16, 25)
after equinox.nn._normalisation: (16, 25)
after equinox.nn._conv: (32, 5)
after jax._src.nn.functions: (32, 5)
after equinox.nn._normalisation: (32, 5)
after jax._src.numpy.lax_numpy: (160,)
after equinox.nn._linear: (80,)
after trapallada.model.nn: (80,)
Encoder(
  layers=[
    Conv1d(
      num_spatial_dims=1,
      weight=f32[8,5,64],
      bias=f32[8,1],
      in_channels=5,
      out_channels=8,
      kernel_size=(64,),
      stride=(2,),
      padding=((0, 0),),
      dilation=(1,),
      groups=1,
      use_bias=True
    ),
    <wrapped function relu>,
    LayerNorm(
      shape=(8, 81),
      eps=1e-05,
      use_weight=True,
      use_bias=True,
      weight=f32[8,81],
      bias=f32[8,81]
    ),
    Conv1d(
      num_spatial_dims=1,
      weight=f32[16,8,32],
      bias=f32[16,1],
      in_channels

# Labels

A class to access the (scarce) available *labels*.

In [22]:
labels = trapallada.data.Labels()
labels

                                   signal        alias
0  0605950101/P0605950101PNX000SRCTSR8001         Javi
1  0840380201/P0840380201PNX000SRCTSR8004  Cataclysmic
2  0690744401/P0690744401PNX000SRCTSR8002            I
3  0148850101/P0148850101PNX000SRCTSR8004           II
4  0740290301/P0740290301PNX000SRCTSR8002          III
5  0720173901/P0720173901PNX000SRCTSR8001           IV

In [23]:
labels.signals

['0605950101/P0605950101PNX000SRCTSR8001',
 '0840380201/P0840380201PNX000SRCTSR8004',
 '0690744401/P0690744401PNX000SRCTSR8002',
 '0148850101/P0148850101PNX000SRCTSR8004',
 '0740290301/P0740290301PNX000SRCTSR8002',
 '0720173901/P0720173901PNX000SRCTSR8001']

In [24]:
labels.signals_without_prefix

['P0605950101PNX000SRCTSR8001',
 'P0840380201PNX000SRCTSR8004',
 'P0690744401PNX000SRCTSR8002',
 'P0148850101PNX000SRCTSR8004',
 'P0740290301PNX000SRCTSR8002',
 'P0720173901PNX000SRCTSR8001']

In [25]:
labels.signals_to_labels

{'0605950101/P0605950101PNX000SRCTSR8001': 'Javi',
 '0840380201/P0840380201PNX000SRCTSR8004': 'Cataclysmic',
 '0690744401/P0690744401PNX000SRCTSR8002': 'I',
 '0148850101/P0148850101PNX000SRCTSR8004': 'II',
 '0740290301/P0740290301PNX000SRCTSR8002': 'III',
 '0720173901/P0720173901PNX000SRCTSR8001': 'IV'}

In [26]:
labels.signals_and_labels

[['0605950101/P0605950101PNX000SRCTSR8001', 'Javi'],
 ['0840380201/P0840380201PNX000SRCTSR8004', 'Cataclysmic'],
 ['0690744401/P0690744401PNX000SRCTSR8002', 'I'],
 ['0148850101/P0148850101PNX000SRCTSR8004', 'II'],
 ['0740290301/P0740290301PNX000SRCTSR8002', 'III'],
 ['0720173901/P0720173901PNX000SRCTSR8001', 'IV']]